# Faruq-v3 — GEO-SHARED60 only (parallel account)

Notebook ini hanya melatih **GEO-SHARED60** untuk seed 42/123/2026. Jalankan GEO-FAM35x3 pada akun Colab lain. Keduanya memakai output Drive yang sama tetapi lock arm/seed berbeda. Test tetap terkunci.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import csv, importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-geometry-conditioning-paired-confirmation-v1/val_reports/geometry_conditioning_paired_three_seed_confirmation.json',
 'experiments/faruq-v3-geometry-family-effect-decomposition-v1/geometry_family_effect_decomposition.json',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/experiment_manifest.json',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/experiment_manifest.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed123_val.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/experiment_manifest.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed2026_val.json')
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,REQUIRED[0]); CONFIRM=require_project_artifact(PROJECT_ROOT,REQUIRED[1]); DECOMP=require_project_artifact(PROJECT_ROOT,REQUIRED[2])
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA_ROOT/'faruq_grouped_summary.json').is_file() and not (DATA_ROOT/'test').exists()
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-geometry-family-factorization-v1'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
ARM='GEO-SHARED60'
print('GPU:',torch.cuda.get_device_name(0)); print('ARM:',ARM); print('OUTPUT:',OUTPUT_ROOT)

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_geometry_family_factorization.py'],cwd=REPO,check=True)
base=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_geometry_family_factorization','--data-root',str(DATA_ROOT),'--project-root',str(PROJECT_ROOT),'--confirmation-summary',str(CONFIRM),'--family-decomposition',str(DECOMP),'--output-root',str(OUTPUT_ROOT),'--device','0']
preflight=OUTPUT_ROOT/'static_preflight.json'
if not preflight.is_file():
    attempt=subprocess.run(base+['--stage','static'],cwd=REPO,text=True,capture_output=True)
    if attempt.returncode!=0:
        print('Preflight sedang dibuat akun lain; menunggu maksimal 5 menit...')
        for _ in range(60):
            if preflight.is_file(): break
            time.sleep(5)
        if not preflight.is_file():
            print(attempt.stdout); print(attempt.stderr); raise RuntimeError('Static preflight gagal')
static=json.loads(preflight.read_text(encoding='utf-8')); assert static['decision']=='PASS'
print('STATIC PASS')

In [ ]:
def progress():
    rows=[]
    for seed in (42,123,2026):
        report=OUTPUT_ROOT/'val_reports'/f'{ARM}_seed{seed}_val.json'; history=OUTPUT_ROOT/f'{ARM}_seed{seed}'/'results.csv'
        if report.is_file(): rows.append(f's{seed}=selesai')
        elif history.is_file():
            with history.open(newline='',encoding='utf-8') as stream: epochs=list(csv.DictReader(stream))
            rows.append(f"s{seed}={epochs[-1]['epoch'] if epochs else '0'}/50")
        else: rows.append(f's{seed}=menunggu')
    return ', '.join(rows)
cmd=base+['--stage','train','--arms',ARM,'--authorize-training']
log=Path('/content/geo_shared60_parallel.log'); print('MULAI',ARM,'| log:',log)
with log.open('w',encoding='utf-8') as stream:
    process=subprocess.Popen(cmd,cwd=REPO,text=True,stdout=stream,stderr=subprocess.STDOUT)
    started=time.monotonic()
    while process.poll() is None:
        print(f'[{(time.monotonic()-started)/60:.1f} menit] {progress()}',flush=True); time.sleep(60)
if process.returncode!=0:
    print('\n'.join(log.read_text(errors='replace').splitlines()[-100:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
print('SELESAI:',progress())

In [ ]:
summary=OUTPUT_ROOT/'val_reports/geometry_family_factorization_three_seed.json'
if summary.is_file():
    result=json.loads(summary.read_text(encoding='utf-8')); print('FINAL SUDAH TERBENTUK:',result['decision'],result['next_action'])
else:
    print(ARM,'selesai. Menunggu GEO-FAM35x3 dari akun lain, lalu jalankan notebook Finalize.')
print('Test tetap terkunci.')